In [11]:
import pandas as pd
import joblib
from collections import deque

In [12]:
MODEL_PATH = "../isolation_forest_model.pkl"
SCALER_PATH = "../scaler.pkl"  # scaler must match the one used during training
THRESHOLD = -0.0518
BUFFER_SIZE = 5
VOTE_THRESHOLD = 3

In [13]:
FEATURE_COLS = ["avg_dwell", "std_dwell", "avg_flight", "std_flight",
                "typing_speed", "backspace_rate",
                "avg_space_dwells", "overlap_rate", "long_pause_rate"]

In [14]:
model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)  # load the same scaler fitted on training data

c:\Users\KARTIKEYA\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\KARTIKEYA\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\KARTIKEYA\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.6.1. This might

In [15]:
def process_window(score, threshold, buffer, vote_threshold):
    is_anomalous_window = score>threshold

    buffer.append(is_anomalous_window)

    recent_anomaly_count = sum(buffer)

    alert_triggered = recent_anomaly_count >= vote_threshold

    return {
        "is_anomalous_window": is_anomalous_window,
        "recent_anomaly_count": recent_anomaly_count,
        "alert_triggered": alert_triggered,
    }

In [16]:
def run_session(features_df, session_name):
    buffer = deque(maxlen=BUFFER_SIZE)

    # Apply the same StandardScaler that was fitted on genuine training data
    scaled = scaler.transform(features_df[FEATURE_COLS])
    scores = -model.decision_function(scaled)

    total_alerts = 0
    first_alert_index = None

    for i, score in enumerate(scores):

        result = process_window(score, THRESHOLD, buffer, VOTE_THRESHOLD)

        if result["alert_triggered"]:
            total_alerts += 1
            if first_alert_index is None:
                first_alert_index = i

    print(f"--- {session_name} ---")
    print(f"Total windows: {len(scores)}")
    print(f"Total alerts: {total_alerts}")
    print(f"First alert at window index: {first_alert_index}")
    print()

    return total_alerts, first_alert_index

In [17]:
test_genuine = pd.read_csv("../data/features.csv")
test_friend = pd.read_csv("../data/features_anomalous.csv")

genuine_alerts, genuine_first = run_session(test_genuine, "Genuine session (should have few/no alerts)")
friend_alerts, friend_first = run_session(test_friend, "Friend session (should alert, ideally early)")

--- Genuine session (should have few/no alerts) ---
Total windows: 1024
Total alerts: 93
First alert at window index: 26

--- Friend session (should alert, ideally early) ---
Total windows: 1011
Total alerts: 933
First alert at window index: 2

